In [ ]:
try:
    df_final = spark.sql("""
    SELECT
      c.player_id,
      c.player_name,
      c.country,
      c.role,
      COUNT(DISTINCT m.match_id)                             AS total_matches,
      SUM(m.runs)                                            AS total_runs,
      SUM(m.balls)                                           AS total_balls,
      ROUND(SUM(m.runs) / NULLIF(SUM(m.balls), 0) * 100, 2) AS overall_strike_rate,
      SUM(m.fours)                                           AS total_fours,
      SUM(m.sixes)                                           AS total_sixes,
      SUM(m.boundary_runs)                                   AS total_boundary_runs,
      SUM(m.wickets)                                         AS total_wickets,
      SUM(m.catches)                                         AS total_catches,
      ROUND(AVG(m.runs), 2)                                  AS avg_runs_per_match,
      MAX(m.runs)                                            AS highest_score,
      CURRENT_TIMESTAMP()                                    AS load_ts
    FROM curated.Cricketers c
    LEFT JOIN curated.matches m
      ON c.player_id = m.player_id
    GROUP BY c.player_id, c.player_name, c.country, c.role
    ORDER BY total_runs DESC
    """)
    df_final.createOrReplaceTempView("player_metrics")
    display(df_final)
    print("✓ Player metrics calculated successfully")
except Exception as e:
    print(f"✗ Error calculating player metrics: {str(e)}")
    raise

In [ ]:
%sql
-- Create consumption delta table if not exists
BEGIN;
  CREATE TABLE IF NOT EXISTS Consumption.player_performance_metrics (
    player_id          INT,
    player_name        STRING,
    country            STRING,
    role               STRING,
    total_matches      BIGINT,
    total_runs         BIGINT,
    total_balls        BIGINT,
    overall_strike_rate DOUBLE,
    total_fours        BIGINT,
    total_sixes        BIGINT,
    total_boundary_runs BIGINT,
    total_wickets      BIGINT,
    total_catches      BIGINT,
    avg_runs_per_match DOUBLE,
    highest_score      INT,
    load_ts            TIMESTAMP
  )
  USING DELTA;

  -- Merge into consumption table
  MERGE INTO Consumption.player_performance_metrics AS target
  USING player_metrics AS source
    ON target.player_id = source.player_id
  WHEN MATCHED THEN
    UPDATE SET
      target.player_name        = source.player_name,
      target.country            = source.country,
      target.role               = source.role,
      target.total_matches      = source.total_matches,
      target.total_runs         = source.total_runs,
      target.total_balls        = source.total_balls,
      target.overall_strike_rate = source.overall_strike_rate,
      target.total_fours        = source.total_fours,
      target.total_sixes        = source.total_sixes,
      target.total_boundary_runs = source.total_boundary_runs,
      target.total_wickets      = source.total_wickets,
      target.total_catches      = source.total_catches,
      target.avg_runs_per_match = source.avg_runs_per_match,
      target.highest_score      = source.highest_score,
      target.load_ts            = source.load_ts
  WHEN NOT MATCHED THEN
    INSERT (
      player_id, player_name, country, role,
      total_matches, total_runs, total_balls, overall_strike_rate,
      total_fours, total_sixes, total_boundary_runs,
      total_wickets, total_catches,
      avg_runs_per_match, highest_score, load_ts
    )
    VALUES (
      source.player_id, source.player_name, source.country, source.role,
      source.total_matches, source.total_runs, source.total_balls, source.overall_strike_rate,
      source.total_fours, source.total_sixes, source.total_boundary_runs,
      source.total_wickets, source.total_catches,
      source.avg_runs_per_match, source.highest_score, source.load_ts
    );
EXCEPTION WHEN OTHERS THEN
  RAISE NOTICE 'Error during merge operation: %', SQLERRM;
  ROLLBACK;
END;